# Experiment 11: Model Monitoring using Prometheus & Grafana

**Objective:**
- Instrument the FastAPI application with Prometheus metrics
- Set up Prometheus to scrape metrics
- Configure Grafana dashboards for monitoring
- Monitor model performance, latency, and request patterns

**Prerequisites:** Run Experiments 1-5 (Model, API, Docker)

## Step 1: Install Required Libraries

In [ ]:
!pip install prometheus-fastapi-instrumentator prometheus-client psutil

## Step 2: Create Instrumented FastAPI Application

In [ ]:
import os

monitored_app_code = '''import pickle
import time
import psutil
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field
from typing import Literal
from prometheus_fastapi_instrumentator import Instrumentator
from prometheus_client import Counter, Histogram, Gauge, Info, Summary

# ── Load Model ──────────────────────────────────────────
with open("model_artifacts/churn_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("model_artifacts/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
with open("model_artifacts/label_encoders.pkl", "rb") as f:
    label_encoders = pickle.load(f)

# ── Custom Prometheus Metrics ──────────────────────────
PREDICTION_COUNT = Counter(
    "model_predictions_total", "Total number of predictions made",
    ["prediction_label"]
)
PREDICTION_LATENCY = Histogram(
    "model_prediction_latency_seconds", "Time taken for model prediction",
    buckets=[0.001, 0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0]
)
CHURN_PROBABILITY = Histogram(
    "model_churn_probability", "Distribution of churn probabilities",
    buckets=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
)
ACTIVE_REQUESTS = Gauge(
    "model_active_requests", "Number of active prediction requests"
)
MODEL_INFO = Info(
    "model_details", "Model metadata"
)
CPU_USAGE = Gauge("system_cpu_percent", "System CPU usage")
MEMORY_USAGE = Gauge("system_memory_percent", "System memory usage")
PREDICTION_ERRORS = Counter(
    "model_prediction_errors_total", "Total prediction errors",
    ["error_type"]
)
FEATURE_VALUES = Summary(
    "model_feature_monthly_charges", "Summary of MonthlyCharges feature values"
)

# ── App Setup ──────────────────────────────────────────
app = FastAPI(title="ML Churn Prediction API - Monitored", version="2.0.0")

# Instrument with default metrics
instrumentator = Instrumentator(
    should_group_status_codes=True,
    should_ignore_untemplated=True,
    excluded_handlers=["/metrics"],
)
instrumentator.instrument(app).expose(app, endpoint="/metrics")

MODEL_INFO.info({
    "model_type": "RandomForestClassifier",
    "framework": "scikit-learn",
    "version": "1.0.0",
    "features": "Gender,SeniorCitizen,Tenure,MonthlyCharges,Contract,PaymentMethod,TotalCharges"
})

# ── Schemas ────────────────────────────────────────────
class ChurnRequest(BaseModel):
    Gender: Literal["Male", "Female"]
    SeniorCitizen: int = Field(ge=0, le=1)
    Tenure: int = Field(ge=0, le=72)
    MonthlyCharges: float = Field(ge=0)
    Contract: Literal["Month-to-month", "One year", "Two year"]
    PaymentMethod: Literal["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"]
    TotalCharges: float = Field(ge=0)

# ── Endpoints ──────────────────────────────────────────
@app.get("/")
def root():
    return {"message": "ML Churn API with Prometheus Monitoring", "metrics": "/metrics"}

@app.get("/health")
def health():
    CPU_USAGE.set(psutil.cpu_percent())
    MEMORY_USAGE.set(psutil.virtual_memory().percent)
    return {"status": "healthy", "cpu": psutil.cpu_percent(), "memory": psutil.virtual_memory().percent}

@app.post("/predict")
def predict(request: ChurnRequest):
    ACTIVE_REQUESTS.inc()
    start_time = time.time()
    try:
        data = request.model_dump()
        FEATURE_VALUES.observe(data["MonthlyCharges"])
        df = pd.DataFrame([data])
        for col in label_encoders:
            if col in df.columns:
                df[col] = label_encoders[col].transform(df[col])
        scaled = scaler.transform(df)
        prediction = model.predict(scaled)[0]
        probabilities = model.predict_proba(scaled)[0]
        label = "Churn" if prediction == 1 else "No Churn"
        
        # Record metrics
        latency = time.time() - start_time
        PREDICTION_LATENCY.observe(latency)
        PREDICTION_COUNT.labels(prediction_label=label).inc()
        CHURN_PROBABILITY.observe(float(probabilities[1]))
        
        return {
            "prediction": int(prediction),
            "prediction_label": label,
            "churn_probability": round(float(probabilities[1]), 4),
            "no_churn_probability": round(float(probabilities[0]), 4),
            "latency_ms": round(latency * 1000, 2)
        }
    except Exception as e:
        PREDICTION_ERRORS.labels(error_type=type(e).__name__).inc()
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        ACTIVE_REQUESTS.dec()

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('monitored_app.py', 'w') as f:
    f.write(monitored_app_code)

print("Created: monitored_app.py (FastAPI with Prometheus metrics)")

## Step 3: Create Prometheus Configuration

In [ ]:
os.makedirs('monitoring', exist_ok=True)

prometheus_config = '''global:
  scrape_interval: 15s
  evaluation_interval: 15s

scrape_configs:
  - job_name: 'ml-api'
    metrics_path: '/metrics'
    scrape_interval: 5s
    static_configs:
      - targets: ['ml-api:8000']
        labels:
          app: 'churn-predictor'
          environment: 'development'

  - job_name: 'prometheus'
    static_configs:
      - targets: ['localhost:9090']
'''

with open('monitoring/prometheus.yml', 'w') as f:
    f.write(prometheus_config)

print("Created: monitoring/prometheus.yml")

## Step 4: Create Grafana Dashboard Configuration

In [ ]:
import json

# Grafana datasource provisioning
os.makedirs('monitoring/grafana/provisioning/datasources', exist_ok=True)
os.makedirs('monitoring/grafana/provisioning/dashboards', exist_ok=True)
os.makedirs('monitoring/grafana/dashboards', exist_ok=True)

datasource_config = '''apiVersion: 1
datasources:
  - name: Prometheus
    type: prometheus
    access: proxy
    url: http://prometheus:9090
    isDefault: true
    editable: true
'''

with open('monitoring/grafana/provisioning/datasources/prometheus.yml', 'w') as f:
    f.write(datasource_config)

dashboard_provisioning = '''apiVersion: 1
providers:
  - name: 'default'
    folder: 'ML Monitoring'
    type: file
    options:
      path: /var/lib/grafana/dashboards
'''

with open('monitoring/grafana/provisioning/dashboards/dashboards.yml', 'w') as f:
    f.write(dashboard_provisioning)

print("Created Grafana provisioning configs")

In [ ]:
# Grafana Dashboard JSON
grafana_dashboard = {
    "dashboard": {
        "id": None,
        "uid": "ml-churn-monitor",
        "title": "ML Churn Model Monitoring",
        "tags": ["ml", "monitoring", "churn"],
        "timezone": "browser",
        "refresh": "10s",
        "time": {"from": "now-1h", "to": "now"},
        "panels": [
            {
                "id": 1, "type": "stat", "title": "Total Predictions",
                "gridPos": {"h": 4, "w": 6, "x": 0, "y": 0},
                "targets": [{"expr": "sum(model_predictions_total)", "refId": "A"}],
                "fieldConfig": {"defaults": {"thresholds": {"steps": [{"color": "blue", "value": None}]}}}
            },
            {
                "id": 2, "type": "stat", "title": "Churn Predictions",
                "gridPos": {"h": 4, "w": 6, "x": 6, "y": 0},
                "targets": [{"expr": 'model_predictions_total{prediction_label="Churn"}', "refId": "A"}],
                "fieldConfig": {"defaults": {"thresholds": {"steps": [{"color": "red", "value": None}]}}}
            },
            {
                "id": 3, "type": "stat", "title": "Avg Latency (ms)",
                "gridPos": {"h": 4, "w": 6, "x": 12, "y": 0},
                "targets": [{"expr": "rate(model_prediction_latency_seconds_sum[5m]) / rate(model_prediction_latency_seconds_count[5m]) * 1000", "refId": "A"}],
                "fieldConfig": {"defaults": {"unit": "ms", "thresholds": {"steps": [{"color": "green", "value": None}, {"color": "yellow", "value": 50}, {"color": "red", "value": 100}]}}}
            },
            {
                "id": 4, "type": "stat", "title": "Error Count",
                "gridPos": {"h": 4, "w": 6, "x": 18, "y": 0},
                "targets": [{"expr": "sum(model_prediction_errors_total)", "refId": "A"}],
                "fieldConfig": {"defaults": {"thresholds": {"steps": [{"color": "green", "value": None}, {"color": "red", "value": 1}]}}}
            },
            {
                "id": 5, "type": "timeseries", "title": "Prediction Rate (per second)",
                "gridPos": {"h": 8, "w": 12, "x": 0, "y": 4},
                "targets": [
                    {"expr": 'rate(model_predictions_total{prediction_label="Churn"}[1m])', "legendFormat": "Churn", "refId": "A"},
                    {"expr": 'rate(model_predictions_total{prediction_label="No Churn"}[1m])', "legendFormat": "No Churn", "refId": "B"}
                ]
            },
            {
                "id": 6, "type": "timeseries", "title": "Prediction Latency",
                "gridPos": {"h": 8, "w": 12, "x": 12, "y": 4},
                "targets": [
                    {"expr": "histogram_quantile(0.50, rate(model_prediction_latency_seconds_bucket[5m]))", "legendFormat": "p50", "refId": "A"},
                    {"expr": "histogram_quantile(0.90, rate(model_prediction_latency_seconds_bucket[5m]))", "legendFormat": "p90", "refId": "B"},
                    {"expr": "histogram_quantile(0.99, rate(model_prediction_latency_seconds_bucket[5m]))", "legendFormat": "p99", "refId": "C"}
                ]
            },
            {
                "id": 7, "type": "histogram", "title": "Churn Probability Distribution",
                "gridPos": {"h": 8, "w": 12, "x": 0, "y": 12},
                "targets": [{"expr": "model_churn_probability_bucket", "refId": "A", "format": "heatmap"}]
            },
            {
                "id": 8, "type": "gauge", "title": "System Resources",
                "gridPos": {"h": 8, "w": 12, "x": 12, "y": 12},
                "targets": [
                    {"expr": "system_cpu_percent", "legendFormat": "CPU %", "refId": "A"},
                    {"expr": "system_memory_percent", "legendFormat": "Memory %", "refId": "B"}
                ],
                "fieldConfig": {"defaults": {"min": 0, "max": 100, "thresholds": {"steps": [{"color": "green", "value": None}, {"color": "yellow", "value": 60}, {"color": "red", "value": 80}]}}}
            },
            {
                "id": 9, "type": "piechart", "title": "Prediction Distribution",
                "gridPos": {"h": 8, "w": 12, "x": 0, "y": 20},
                "targets": [{"expr": "model_predictions_total", "legendFormat": "{{prediction_label}}", "refId": "A"}]
            },
            {
                "id": 10, "type": "timeseries", "title": "HTTP Request Rate",
                "gridPos": {"h": 8, "w": 12, "x": 12, "y": 20},
                "targets": [
                    {"expr": "rate(http_requests_total[1m])", "legendFormat": "{{method}} {{handler}} {{status}}", "refId": "A"}
                ]
            }
        ],
        "schemaVersion": 38
    }
}

with open('monitoring/grafana/dashboards/ml_monitoring.json', 'w') as f:
    json.dump(grafana_dashboard, f, indent=2)

print("Created: monitoring/grafana/dashboards/ml_monitoring.json")

## Step 5: Create Docker Compose for Full Monitoring Stack

In [ ]:
docker_compose_monitoring = '''version: "3.8"

services:
  ml-api:
    build: .
    container_name: ml-api-monitored
    ports:
      - "8000:8000"
    command: uvicorn monitored_app:app --host 0.0.0.0 --port 8000
    volumes:
      - ./model_artifacts:/app/model_artifacts
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
    networks:
      - monitoring

  prometheus:
    image: prom/prometheus:latest
    container_name: prometheus
    ports:
      - "9090:9090"
    volumes:
      - ./monitoring/prometheus.yml:/etc/prometheus/prometheus.yml
      - prometheus_data:/prometheus
    command:
      - "--config.file=/etc/prometheus/prometheus.yml"
      - "--storage.tsdb.retention.time=7d"
    depends_on:
      - ml-api
    networks:
      - monitoring

  grafana:
    image: grafana/grafana:latest
    container_name: grafana
    ports:
      - "3000:3000"
    environment:
      - GF_SECURITY_ADMIN_USER=admin
      - GF_SECURITY_ADMIN_PASSWORD=admin123
      - GF_USERS_ALLOW_SIGN_UP=false
    volumes:
      - ./monitoring/grafana/provisioning:/etc/grafana/provisioning
      - ./monitoring/grafana/dashboards:/var/lib/grafana/dashboards
      - grafana_data:/var/lib/grafana
    depends_on:
      - prometheus
    networks:
      - monitoring

volumes:
  prometheus_data:
  grafana_data:

networks:
  monitoring:
    driver: bridge
'''

with open('docker-compose-monitoring.yml', 'w') as f:
    f.write(docker_compose_monitoring)

print("Created: docker-compose-monitoring.yml")
print("\nServices:")
print("  - ML API:      http://localhost:8000")
print("  - Prometheus:   http://localhost:9090")
print("  - Grafana:      http://localhost:3000 (admin/admin123)")

## Step 6: Update Dockerfile for Monitoring Dependencies

In [ ]:
monitoring_requirements = '''fastapi==0.104.1
uvicorn==0.24.0
pydantic==2.5.0
scikit-learn==1.3.2
pandas==2.1.3
numpy==1.26.2
python-jose[cryptography]==3.3.0
passlib[bcrypt]==1.7.4
python-multipart==0.0.6
python-json-logger==2.0.7
prometheus-fastapi-instrumentator==6.1.0
prometheus-client==0.19.0
psutil==5.9.6
'''

with open('requirements-monitoring.txt', 'w') as f:
    f.write(monitoring_requirements)

print("Created: requirements-monitoring.txt")

## Step 7: Test the Monitoring Setup Locally

In [ ]:
import threading
import time
import requests
import uvicorn
import nest_asyncio

nest_asyncio.apply()

# Import the monitored app
import importlib
import sys
if 'monitored_app' in sys.modules:
    del sys.modules['monitored_app']
import monitored_app

# Start the server
server_thread = threading.Thread(
    target=uvicorn.run,
    args=(monitored_app.app,),
    kwargs={"host": "127.0.0.1", "port": 8000, "log_level": "warning"},
    daemon=True
)
server_thread.start()
time.sleep(3)
print("Server started on http://127.0.0.1:8000")

In [ ]:
# Send sample predictions to generate metrics
import random

base_url = "http://127.0.0.1:8000"

test_customers = [
    {"Gender": "Male", "SeniorCitizen": 0, "Tenure": 48, "MonthlyCharges": 45.0, "Contract": "Two year", "PaymentMethod": "Bank transfer (automatic)", "TotalCharges": 2160.0},
    {"Gender": "Female", "SeniorCitizen": 1, "Tenure": 2, "MonthlyCharges": 95.0, "Contract": "Month-to-month", "PaymentMethod": "Electronic check", "TotalCharges": 190.0},
    {"Gender": "Male", "SeniorCitizen": 0, "Tenure": 24, "MonthlyCharges": 60.0, "Contract": "One year", "PaymentMethod": "Mailed check", "TotalCharges": 1440.0},
    {"Gender": "Female", "SeniorCitizen": 0, "Tenure": 6, "MonthlyCharges": 80.0, "Contract": "Month-to-month", "PaymentMethod": "Electronic check", "TotalCharges": 480.0},
    {"Gender": "Male", "SeniorCitizen": 1, "Tenure": 60, "MonthlyCharges": 30.0, "Contract": "Two year", "PaymentMethod": "Credit card (automatic)", "TotalCharges": 1800.0},
]

print("Sending 20 prediction requests...")
for i in range(20):
    customer = random.choice(test_customers)
    resp = requests.post(f"{base_url}/predict", json=customer)
    result = resp.json()
    if i < 5:  # Print first 5
        print(f"  Request {i+1}: {result['prediction_label']} (prob: {result['churn_probability']:.4f}, latency: {result['latency_ms']}ms)")

print(f"\n✅ Sent 20 requests")

In [ ]:
# Fetch and display Prometheus metrics
resp = requests.get(f"{base_url}/metrics")
metrics_text = resp.text

# Parse key metrics
print("=" * 60)
print("PROMETHEUS METRICS ENDPOINT (/metrics)")
print("=" * 60)

key_metrics = [
    'model_predictions_total',
    'model_prediction_latency_seconds',
    'model_churn_probability',
    'model_prediction_errors_total',
    'system_cpu_percent',
    'system_memory_percent',
    'model_details_info',
]

for line in metrics_text.split('\n'):
    for metric in key_metrics:
        if line.startswith(metric) and not line.startswith('#'):
            print(f"  {line}")
            break

print(f"\nTotal metrics lines: {len(metrics_text.split(chr(10)))}")

## Step 8: Create Alert Rules for Prometheus

In [ ]:
alert_rules = '''groups:
  - name: ml_model_alerts
    rules:
      - alert: HighChurnRate
        expr: |
          sum(rate(model_predictions_total{prediction_label="Churn"}[5m]))
          / sum(rate(model_predictions_total[5m])) > 0.5
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "High churn prediction rate"
          description: "More than 50% of predictions are Churn for 5+ minutes"

      - alert: HighLatency
        expr: |
          histogram_quantile(0.95, rate(model_prediction_latency_seconds_bucket[5m])) > 0.5
        for: 2m
        labels:
          severity: critical
        annotations:
          summary: "High prediction latency"
          description: "95th percentile latency > 500ms"

      - alert: PredictionErrors
        expr: sum(rate(model_prediction_errors_total[5m])) > 0.1
        for: 1m
        labels:
          severity: critical
        annotations:
          summary: "Model prediction errors detected"

      - alert: APIDown
        expr: up{job="ml-api"} == 0
        for: 1m
        labels:
          severity: critical
        annotations:
          summary: "ML API is down"

      - alert: HighCPU
        expr: system_cpu_percent > 80
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "High CPU usage ({{ $value }}%)"

      - alert: HighMemory
        expr: system_memory_percent > 85
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "High memory usage ({{ $value }}%)"
'''

with open('monitoring/alert_rules.yml', 'w') as f:
    f.write(alert_rules)

print("Created: monitoring/alert_rules.yml")
print("\nAlerts configured:")
print("  - HighChurnRate (>50% churn predictions for 5m)")
print("  - HighLatency (p95 > 500ms for 2m)")
print("  - PredictionErrors (errors detected for 1m)")
print("  - APIDown (API unreachable for 1m)")
print("  - HighCPU (>80% for 5m)")
print("  - HighMemory (>85% for 5m)")

## Step 9: Summary & Launch Commands

In [ ]:
print("=" * 60)
print("MONITORING STACK SUMMARY")
print("=" * 60)
print("")
print("Files created:")
print("  - monitored_app.py          (FastAPI + Prometheus metrics)")
print("  - monitoring/prometheus.yml  (Prometheus config)")
print("  - monitoring/alert_rules.yml (Alert rules)")
print("  - monitoring/grafana/        (Grafana provisioning + dashboards)")
print("  - docker-compose-monitoring.yml")
print("  - requirements-monitoring.txt")
print("")
print("To launch the full monitoring stack:")
print("  docker-compose -f docker-compose-monitoring.yml up --build")
print("")
print("Access points:")
print("  ML API:     http://localhost:8000")
print("  Metrics:    http://localhost:8000/metrics")
print("  Prometheus: http://localhost:9090")
print("  Grafana:    http://localhost:3000 (admin / admin123)")
print("")
print("Custom metrics exposed:")
print("  - model_predictions_total (counter by label)")
print("  - model_prediction_latency_seconds (histogram)")
print("  - model_churn_probability (histogram)")
print("  - model_prediction_errors_total (counter)")
print("  - model_active_requests (gauge)")
print("  - system_cpu_percent (gauge)")
print("  - system_memory_percent (gauge)")
print("  - model_details_info (info)")
print("\n✅ Experiment 11 Complete!")